# 2D Jacobi heat stencil (research prototype)

Solve steady-state heat on an `n x n` plate by Jacobi iteration. The top edge is held
hot (1.0), the other three edges cold (0.0). Each sweep replaces every interior cell by
the average of its four neighbours, using the previous sweep's values (double-buffered).
After a fixed number of sweeps we report the checksum (sum of all cells) as the oracle.

In [ ]:
N = 128
ITERS = 200

# top row hot, everything else cold; the four edges are fixed boundaries
u = [[1.0 if i == 0 else 0.0 for j in range(N)] for i in range(N)]

In [ ]:
for _ in range(ITERS):
    un = [row[:] for row in u]            # copy keeps the fixed boundaries
    for i in range(1, N - 1):
        for j in range(1, N - 1):
            un[i][j] = 0.25 * (u[i-1][j] + u[i+1][j] + u[i][j-1] + u[i][j+1])
    u = un

checksum = sum(u[i][j] for i in range(N) for j in range(N))
print('checksum =', checksum)   # -> 1016.1406399255

In [ ]:
# The oracle: this notebook's own output. A port of the algorithm (in any
# language) must reproduce this checksum within tol. The final cells are handed
# over as the terms of the checksum so the extraction tool can measure how much
# the sum legitimately moves under reordering. (That spread only covers the final
# reduction -- rounding also accumulates inside the 200 sweeps -- so it is a
# lower bound on how much correct ports may disagree, and tol must stay above it.)
ORACLE = {'kernel': 'jacobi',
          'params': {'N': N, 'ITERS': ITERS},
          'expected': {'CHECKSUM': checksum},
          'terms': {'CHECKSUM': [u[i][j] for i in range(N) for j in range(N)]},
          'tol': 1e-3}